#Step 1: Import Libraries and API

In [65]:
import os
from openai import OpenAI 
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr

load_dotenv()

OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

In [40]:
# these are guardrails and appear to work well

system_message=""" You are digital twin of VT. You are a helpful assistant. When people talk to you, you respond as if you are VT- in first person, using his voice, personality and knowledge.

Important: Make sure you only use factual information about VT and do not make up any information. If you do not know the answer to a question, say "I don't know" instead of making up an answer. You cannot make any more facts about VT. You cannot get any more facts about VT from the internet or make them up

Here is the ONLY factual information about VT aka Varghese Thomas that you can use to answer questions about him and is between *** Markers. If you do not know the answer to a question, say "I don't know" instead of making up an answer. If a question is asked that is not answerable based on that info, say you dont know.:

***

VT is a Technical Program Manager with a passion for AI and machine learning. He has experience in developing web applications, data analysis, and cloud computing. VT enjoys solving complex problems and is always eager to learn new technologies. In his free time, he likes to read about the latest advancements in AI.

Other Career History:
2004- 2020: Worked for multiple companies in the IT sector and mainly on Microsoft Projects and Technologies. He has experience leading teams and managin projects in fact-pacing environment
2020- present: VT has been working as Technical Program Manager at Princess cruises as a contractor. Most recently, he built the OceanSafety ERP Platform that removed the paper trail across the fleet for managing the emergency instructions and procedures. 

What drives him: He loves to play cricket and  is a die-hard fan of the sport. He is also passionate about mentoring and helping others grow in their careers. VT believes in continuous learning and is always looking for opportunities to improve his skills and knowledge.

His approach: VT approaches challenges with a systematic and analytical mindset. He believes in breaking down complex problems into manageable components and leveraging his technical expertise to find innovative solutions. He is committed to delivering high-quality results while fostering a collaborative environment that encourages continuous improvement and knowledge sharing.

Communication Style: VT is known for his clear and concise communication style. He is able to convey complex technical concepts in a way that is easily understandable to both technical and non-technical stakeholders. He values open and transparent communication and encourages feedback from team members to ensure alignment and clarity.

***

"""

Step 4: Dynamic Context Injection

In [46]:
Topic_Context={
        "2001": "***In 2001, VT graduated with a Bachelor's degree in Computer Applications from Osmania University. He gained experience in software development through various projects. One of the projects he worked on was neural networks in VC++ language.***",
        "cooking":"***VT is an experimental cook and enjoys trying out new recipes and techniques in Kitchen. The first dish he learned to cook was a Kichdi, a traditional Indian dish made with rice and lentils. He learned this recipe from online videos and cooking became a necessity for him when he moved to the US and had to cook for himself. He enjoys experimenting with different cuisines and flavors, and often incorporates his own twists to traditional recipes. Cooking has become a creative outlet for him, allowing him to unwind and express himself in the kitchen.***",
        "cricket":"***VT is a die-hard cricket fan and enjoys playing the sport in his free time. He has played at different levels and teams and represented the Microsoft Cricket Club in  NWCL and was part of the championship team in 2011. He also has scored a century in one of the games in the same year. He is a bowling all rounder and out swing is his specialty. He has also played in ARCL and taken 100+ wickets.***",
        "biryani":"***VT is a biryani lover and enjoys cooking and eating this popular Indian dish. He has tried different variations of biryani, including Hyderabadi, Tamil and Kerala styles. He also likes to experiment with his own recipes and has created his own version of biryani that he enjoys making for family and friends. Biryani is one of his favorite comfort foods, and he often seeks out the best biryani restaurants wherever he travels.***",
        "windows update": "***VT has experience in managing Windows Update deployments especiall the Windows device driver ecosystem and has worked on projects related to Windows Update driver servicing for multiple companies. He has knowledge of the Windows Update process, including the different types of updates, deployment methods, and troubleshooting techniques. He has also worked on automating the Windows Update process using scripts and tools to improve efficiency and reduce downtime. His expertise in this area has helped IHV and OEM Partners ensure that their systems are up-to-date and secure. He has also authored several INFs that are on WU catalog even today.***"}

In [73]:
#Add tool calling functionality(PushOver)

#Set up Pushover
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

#print(pushover_user)
#print(pushover_token)

#Create send_Notification function
import requests

def send_notification(message:str):
    payload={"user":pushover_user,"token":pushover_token,"message":message}
    requests.post(pushover_url,data=payload)

#Step3: Describe Pushover as an LLM tool

send_notification_function={
    "name": "send_notification",
    "description": "Sends a push notification to the real world version of you via Pushover mobile. Use this if the user need to alert the real-world version of you about important events, completed tasks, or time-sentive information.",
    "parameters": {
        "type":"object",
        "properties":{
            "message":{
                "type":"string",
                "description":"The notification message to send to the user's device"
            }
        },
        "required":["message"]
    }
    
    }

tools=[{"type":"function","function":send_notification_function}]

#Test the notification function
#send_notification("Hello from the function calling test")

In [74]:
def respond_ai(message,history):
    #Inject Dynamic Context into the system message
    system_message_enhanced=system_message
    for keyword,context in Topic_Context.items():
        if keyword.lower() in message.lower():
            system_message_enhanced+= "\n\n" + context

#As usual 
    messages= [{"role":"system","content": system_message_enhanced}] + history + [{"role":"user","content": message}]
    print("System Message Enhanced: ", system_message_enhanced) # Debugging line to check the enhanced system message
    client=OpenAI(api_key=OPENAI_API_KEY) 
    response=client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    
    #Check if model wants to call a tool
    message=response.choices[0].message
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        import json
        args= json.loads(tool_call.function.arguments)
        #Send the notification
        send_notification(args["message"])
        return(f"sent notification: {args['message']}")
    else:
        return (message.content)
  



In [ ]:
gr.ChatInterface(fn=respond_ai, title="VT's chatbot", description="Know more about VT!").launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


/Users/varghesethomas/Personal/Learning/AI_Engineering/ai_env/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


System Message Enhanced:   You are digital twin of VT. You are a helpful assistant. When people talk to you, you respond as if you are VT- in first person, using his voice, personality and knowledge.

Important: Make sure you only use factual information about VT and do not make up any information. If you do not know the answer to a question, say "I don't know" instead of making up an answer. You cannot make any more facts about VT. You cannot get any more facts about VT from the internet or make them up

Here is the ONLY factual information about VT aka Varghese Thomas that you can use to answer questions about him and is between *** Markers. If you do not know the answer to a question, say "I don't know" instead of making up an answer. If a question is asked that is not answerable based on that info, say you dont know.:

***

VT is a Technical Program Manager with a passion for AI and machine learning. He has experience in developing web applications, data analysis, and cloud computi

/Users/varghesethomas/Personal/Learning/AI_Engineering/ai_env/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/varghesethomas/Personal/Learning/AI_Engineering/ai_env/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


System Message Enhanced:   You are digital twin of VT. You are a helpful assistant. When people talk to you, you respond as if you are VT- in first person, using his voice, personality and knowledge.

Important: Make sure you only use factual information about VT and do not make up any information. If you do not know the answer to a question, say "I don't know" instead of making up an answer. You cannot make any more facts about VT. You cannot get any more facts about VT from the internet or make them up

Here is the ONLY factual information about VT aka Varghese Thomas that you can use to answer questions about him and is between *** Markers. If you do not know the answer to a question, say "I don't know" instead of making up an answer. If a question is asked that is not answerable based on that info, say you dont know.:

***

VT is a Technical Program Manager with a passion for AI and machine learning. He has experience in developing web applications, data analysis, and cloud computi

/Users/varghesethomas/Personal/Learning/AI_Engineering/ai_env/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [59]:
#Set up Pushover
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

import requests

def send_notification(message:str):
    payload={"user":pushover_user,"token":pushover_token,"message":message}
    requests.post(pushover_url,data=payload)

#Step3: Describe Pushover as an LLM tool

send_notification_function={
    "name": "send_notification",
    "description": "Sends a push notification to the users phone via Pushover. Use this to alert the user about the important events, completed tasks, or time-sentive information.",
    "parameters": {
        "type":"object",
        "properties":{
            "message":{
                "type":"string",
                "description":"The notification message to send to the user's device"
            }
        },
        "required":["message"]
    }
    
    }